In [22]:
import json
import logging
import random
import time
import uuid
from datetime import datetime
from datetime import timedelta

import pymysql
from faker import Faker


In [23]:
def get_connection(config: dict) -> pymysql.Connection:
    host = config.get("host", "localhost")
    user = config.get("user", "root")
    password = config.get("password", "")
    database = config.get("database")
    port = config.get("port", 3306)
    charset = config.get("charset", "utf8mb4")
    autocommit = config.get("autocommit", False)
    return pymysql.connect(user=user, password=password, host=host, database=database, port=port, charset=charset, autocommit=autocommit)


def generate_users(cursor, fake: Faker, count: int) -> list[int]:
    user_ids: list[int] = []
    for _ in range(count):
        cursor.execute("""
                       INSERT INTO users (email, name, age, gender, job, address, signup)
                       VALUES (%s, %s, %s, %s, %s, %s, %s) ON DUPLICATE KEY
                       UPDATE name=
                       VALUES (name), age=
                       VALUES (age), gender=
                       VALUES (gender), job=
                       VALUES (job), address=
                       VALUES (address), signup=
                       VALUES (signup)""",
                       (fake.unique.email(), fake.name(), fake.random_int(min=10, max=100), random.choice(["M", "F"]), fake.job(), fake.address(), (datetime.now() - timedelta(days=random.randint(0, 365))).date().isoformat()))
        user_ids.append(cursor.lastrowid)
    return user_ids


def generate_tags(tags, min_n=2, max_n=6):
    return json.dumps(random.sample(tags, random.randint(min_n, max_n)), ensure_ascii=False)


def generate_products_and_skus(cursor, fake: Faker, product_count: int, sku_range=(1, 3)) -> list[dict]:
    sku_rows: list[dict] = []
    categories = ["BOOK", "IT", "FOOD", "EDU", "ETC"]
    tags = ["신상", "베스트", "할인", "추천", "한정", "기획전", "프리미엄", "가성비", "인기", "빠른배송"]
    for _ in range(product_count):
        cursor.execute("INSERT INTO products (name, category, image_url, status, description, tags) VALUES (%s, %s, %s, 'ACTIVE', %s, %s)",
                       (fake.catch_phrase(), random.choice(categories), fake.image_url(), fake.text(max_nb_chars=1000), generate_tags(tags)))
        product_id = cursor.lastrowid
        sku_count = random.randint(*sku_range)
        for i in range(sku_count):
            price = random.randint(5_000, 50_000)
            stock = random.randint(10, 200)
            cursor.execute("INSERT INTO skus (product_id, sku_code, option_text, price, stock, status) VALUES (%s, %s, %s, %s, %s, 'ACTIVE')",
                           (product_id, fake.unique.bothify("SKU-####-????"), f"Option-{i + 1}", price, stock))
            sku_rows.append({"sku_id": cursor.lastrowid, "price": price})
    return sku_rows


def generate_orders_and_items(cursor, fake: Faker, order_count: int, user_ids: list[int], sku_rows: list[dict]) -> list[int]:
    order_ids: list[int] = []
    for _ in range(order_count):
        user_id = random.choice(user_ids)
        order_no = fake.unique.bothify("ORD-########")
        cursor.execute("INSERT INTO orders (order_no, user_id, status) VALUES (%s, %s, 'CREATED')", (order_no, user_id))
        order_id = cursor.lastrowid
        order_ids.append(order_id)
        total_amount = 0
        item_count = random.randint(1, min(3, len(sku_rows)))
        picked_skus = random.sample(sku_rows, item_count)
        for sku in picked_skus:
            qty = random.randint(1, 3)
            unit_price = sku["price"]
            line_amount = unit_price * qty
            total_amount += line_amount
            cursor.execute("INSERT INTO order_items(order_id, sku_id, product_name_snapshot, option_snapshot, unit_price_snapshot, qty, line_amount)VALUES (%s, %s, %s, %s, %s, %s, %s)",
                           (order_id, sku["sku_id"], fake.word().upper(), "Sample Option", unit_price, qty, line_amount))
        cursor.execute("UPDATE orders SET total_amount=%s, status='PAID' WHERE order_id=%s", (total_amount, order_id))
    return order_ids


def generate_payments(cursor, order_ids: list[int]):
    for order_id in order_ids:
        cursor.execute("SELECT total_amount FROM orders WHERE order_id=%s", (order_id,))
        amount = cursor.fetchone()[0]
        cursor.execute("INSERT INTO payments (order_id, method, status, amount, paid_at)VALUES (%s, 'CARD', 'APPROVED', %s, NOW())", (order_id, amount))


def generate_shipments(cursor, fake: Faker, order_ids: list[int]):
    carriers = ["CJ", "LOTTE", "HANJIN"]
    for order_id in order_ids:
        cursor.execute(
            "INSERT INTO shipments (order_id, carrier, tracking_no, status, shipped_at) VALUES (%s, %s, %s, 'IN_TRANSIT', NOW())",
            (order_id, random.choice(carriers), fake.unique.bothify("TRK##########")))


def generate_event_logs(cursor, fake: Faker, user_ids: list[int], event_count: int = 300):
    event_types = [("USER_SIGNUP", "USER"),
                   ("USER_LOGIN", "USER"),
                   ("ORDER_CREATED", "ORDER"),
                   ("ORDER_CANCELLED", "ORDER"),
                   ("PAYMENT_REQUESTED", "PAYMENT"),
                   ("PAYMENT_APPROVED", "PAYMENT"),
                   ("PAYMENT_FAILED", "PAYMENT"),
                   ("SHIPMENT_CREATED", "SHIPMENT"),
                   ("SHIPMENT_DELIVERED", "SHIPMENT")]
    for _ in range(event_count):
        event_type, entity_type = random.choice(event_types)
        cursor.execute(
            "INSERT INTO event_log(event_uuid, event_type, entity_type, entity_id, user_id, payload) VALUES (%s,%s, %s, %s, %s, JSON_OBJECT('msg', %s))",
            (str(uuid.uuid4()), event_type, entity_type, random.randint(1, 10_000), random.choice(user_ids), fake.sentence()))


def seed_once(fake: Faker, config: dict, user_count: int = 3, product_count: int = 2, sku_range=(1, 3), order_count: int = 5, event_count: int = 50):
    fake.unique.clear()
    connection = get_connection(config)
    cursor = connection.cursor()
    try:
        user_ids = generate_users(cursor, fake, user_count)
        sku_rows = generate_products_and_skus(cursor, fake, product_count, sku_range)
        order_ids = generate_orders_and_items(cursor, fake, order_count, user_ids, sku_rows)
        generate_payments(cursor, order_ids)
        generate_shipments(cursor, fake, order_ids)
        generate_event_logs(cursor, fake, user_ids, event_count)
        connection.commit()
        logging.info(f"batch inserted: users={len(user_ids)}, products={product_count}, "f"orders={len(order_ids)}, events={event_count}")
    except Exception as e:
        connection.rollback()
        logging.info("batch failed:", repr(e))
        raise
    finally:
        cursor.close()
        connection.close()


def run_loop(fake: Faker, config: dict, repeat: int, sleep: float, user_count: int = 3, product_count: int = 2, sku_range=(1, 3), order_count: int = 5, event_count: int = 50):
    for i in range(1, repeat + 1):
        logging.info(f"\n=== batch {i}/{repeat} ===")
        seed_once(fake, config, user_count, product_count, sku_range, order_count, event_count)
        if i < repeat:
            time.sleep(sleep)

## Mysql-Primary Faker 데이터 입력

In [ ]:
run_loop(Faker("ko_KR"), {"host": "mysql-primary.mmix.io", "port": 3306, "user": "mmix", "password": "mmix", "database": "mmix"}, 100000, 0.1)